## 14. Attack 4 — Iterative Noise-Denoise Perturbation

Iteratively:
1. Add Gaussian noise: $x_{noisy} = x + \mathcal{N}(0, \sigma^2)$
2. Denoise using median filter + Gaussian blur

Tests sensitivity to local pixel corruption via iterative noise-denoise cycles.
Unlike diffusion models, no learned generative prior is used.

**Goal:** Measure how prediction stability degrades under repeated noise-denoise cycles.

## 1. Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple
from collections import defaultdict

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from scipy.ndimage import median_filter
from scipy.ndimage import gaussian_filter

# Dark theme for paper-quality figures
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': 'black', 'axes.facecolor': 'black',
    'savefig.facecolor': 'black', 'text.color': 'white',
    'axes.labelcolor': 'white', 'xtick.color': 'white',
    'ytick.color': 'white', 'legend.facecolor': 'black',
    'legend.edgecolor': 'white',
})

print(f"TensorFlow: {tf.__version__}")
print(f"Eager execution: {tf.executing_eagerly()}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONFIGURATION  (matches training & attack notebooks exactly)
# ════════════════════════════════════════════════════════════════

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1

# Paths (relative to this notebook inside Model Training/)
CHECKPOINT_DIR = Path('checkpoints')
DATA_DIR = Path('caltech101_data')
SPLIT_FILE = Path('frozen_split_indices.json')
BASELINES_FILE = Path('clean_baselines/clean_baselines.json')
RESULTS_DIR = Path('genai_results')
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

MODEL_NAMES = ['VGG19', 'ResNet50', 'DenseNet121']

PREPROCESS_FNS = {
    'VGG19': vgg_preprocess,
    'ResNet50': resnet_preprocess,
    'DenseNet121': densenet_preprocess,
}
PREPROCESS_MODE = {
    'VGG19': 'caffe',
    'ResNet50': 'caffe',
    'DenseNet121': 'torch',
}

# Attack hyperparameters
INTERP_ALPHAS = [0.1, 0.2, 0.3, 0.4]
PROTO_LAMBDAS = [0.2, 0.4, 0.6]
DIFFUSION_STEPS = 5
DIFFUSION_SIGMA = 0.05  # Gaussian noise std per step

# Feature inversion optimisation
INVERSION_STEPS = 100
INVERSION_LR = 0.01

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Configuration loaded.")
print(f"Models: {MODEL_NAMES}")
print(f"Interpolation alphas: {INTERP_ALPHAS}")
print(f"Prototype lambdas: {PROTO_LAMBDAS}")

## 3. Load Frozen Test Split

In [ ]:
# ════════════════════════════════════════════════════════════════
# REPRODUCE EXACT TEST SPLIT from training notebook
# ════════════════════════════════════════════════════════════════

def find_image_root(base_path: Path) -> Path:
    """Find 101_ObjectCategories directory."""
    for obj_cat in base_path.rglob('101_ObjectCategories'):
        if obj_cat.is_dir():
            subdirs = [d for d in obj_cat.iterdir() if d.is_dir() and d.name != '__MACOSX']
            if len(subdirs) > 10:
                return obj_cat
    raise FileNotFoundError(f"101_ObjectCategories not found under {base_path}")

IMAGE_ROOT = find_image_root(DATA_DIR)

# Get class names (same logic as training notebook)
exclude_dirs = {'__MACOSX', '.DS_Store', 'BACKGROUND_Google', '__pycache__'}
CLASS_NAMES = sorted([
    d.name for d in IMAGE_ROOT.iterdir()
    if d.is_dir() and d.name not in exclude_dirs and not d.name.startswith('.')
])
NUM_CLASSES = len(CLASS_NAMES)
class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

# Collect all paths deterministically (sorted)
all_paths, all_labels = [], []
for class_name in CLASS_NAMES:
    class_dir = IMAGE_ROOT / class_name
    for img_path in sorted(class_dir.iterdir()):
        if img_path.is_file() and img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.gif', '.bmp']:
            all_paths.append(str(img_path))
            all_labels.append(class_to_idx[class_name])
all_paths = np.array(all_paths)
all_labels = np.array(all_labels)

# Load frozen indices
assert SPLIT_FILE.exists(), f"Frozen split file not found: {SPLIT_FILE}"
with open(SPLIT_FILE, 'r') as f:
    saved = json.load(f)
indices = np.array(saved['indices'])
assert saved['seed'] == SEED, f"Seed mismatch: file={saved['seed']}, expected={SEED}"

all_paths = all_paths[indices]
all_labels = all_labels[indices]

train_size = int(TRAIN_SPLIT * len(all_paths))
val_size = int(VAL_SPLIT * len(all_paths))

test_paths = all_paths[train_size + val_size:]
test_labels = all_labels[train_size + val_size:]

print(f"Image root: {IMAGE_ROOT}")
print(f"Classes: {NUM_CLASSES}")
print(f"Test samples: {len(test_paths)}")
print(f"Frozen split seed: {saved['seed']}")

# Cross-check with baselines
with open(BASELINES_FILE, 'r') as f:
    baselines = json.load(f)
assert baselines['test_samples'] == len(test_paths), \
    f"Test count mismatch: baselines={baselines['test_samples']}, computed={len(test_paths)}"
print(f"\u2705 Test split matches clean_baselines.json ({len(test_paths)} samples)")

## 4. Build Deterministic Test Dataset

In [ ]:
def build_raw_test_dataset(paths: np.ndarray, labels: np.ndarray,
                           img_size: Tuple[int, int],
                           batch_size: int) -> tf.data.Dataset:
    """Build deterministic test dataset yielding RAW [0,1] images + labels.
    No augmentation, no shuffle, no model-specific preprocessing."""
    def load_image(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

raw_test_ds = build_raw_test_dataset(test_paths, test_labels, IMG_SIZE, BATCH_SIZE)

# Verify first batch
for imgs, labs in raw_test_ds.take(1):
    print(f"Batch shape: {imgs.shape}, dtype: {imgs.dtype}")
    print(f"Pixel range: [{imgs.numpy().min():.3f}, {imgs.numpy().max():.3f}]")
    print(f"Labels shape: {labs.shape}, sample: {labs[:5].numpy()}")
print("\n\u2705 Raw test dataset built (deterministic, [0,1], no augmentation)")

## 5. Preprocessing Adapters

In [ ]:
def preprocess_for_model(images_01: tf.Tensor, model_name: str) -> tf.Tensor:
    """Convert [0,1] images to model-specific input format.
    VGG19/ResNet50 (caffe): [0,1] -> [0,255] -> BGR -> mean subtraction
    DenseNet121 (torch):    [0,1] -> normalize(mean, std)
    """
    if model_name in ('VGG19', 'ResNet50'):
        x = images_01 * 255.0
        x = x[..., ::-1]  # RGB -> BGR
        mean = tf.constant([103.939, 116.779, 123.68], dtype=tf.float32)
        x = x - mean
    else:  # DenseNet121 — torch mode
        mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
        std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
        x = (images_01 - mean) / std
    return x

print("\u2705 Preprocessing adapters defined.")

## 6. Load Trained Models

In [ ]:
models: Dict[str, keras.Model] = {}

for name in MODEL_NAMES:
    ckpt = CHECKPOINT_DIR / f"{name}_best.h5"
    assert ckpt.exists(), f"Checkpoint not found: {ckpt}"
    model = keras.models.load_model(ckpt)
    models[name] = model
    params = model.count_params()
    clean_acc = baselines['models'][name]['accuracy']
    print(f"{name}: {params:,} params | clean acc = {clean_acc*100:.2f}% | loaded from {ckpt}")

print(f"\n\u2705 All {len(models)} models loaded. Inference only — no training.")

## 7. Build Feature Extractors (Penultimate Layer)

For each model, create a sub-model that outputs the penultimate-layer feature vector (before the final classification head).  
This is used by the Latent Interpolation and Prototype Shift attacks.

In [ ]:
feature_extractors: Dict[str, keras.Model] = {}

for name, model in models.items():
    # The penultimate layer is the GlobalAveragePooling / Flatten output
    # before the final Dense(101) classification head.
    # For typical fine-tuned models: input -> base -> GAP -> Dense(101)
    # We want the GAP output.
    penultimate_layer = model.layers[-2]  # layer before final Dense
    feat_model = keras.Model(
        inputs=model.input,
        outputs=penultimate_layer.output,
        name=f"{name}_features"
    )
    feature_extractors[name] = feat_model
    print(f"{name}: feature dim = {penultimate_layer.output_shape[-1]} "
          f"(layer: {penultimate_layer.name})")

print(f"\n\u2705 Feature extractors built for all {len(feature_extractors)} models.")

## 8. Extract Test Set Features & Identify Correctly Classified Samples

For each model:
- Extract penultimate features for every test image.
- Record the predicted label.
- Only correctly classified images are used for attacks.

In [ ]:
# Storage: features, predictions, correct masks per model
test_features: Dict[str, np.ndarray] = {}
test_preds: Dict[str, np.ndarray] = {}
correct_masks: Dict[str, np.ndarray] = {}

for name in MODEL_NAMES:
    print(f"\nExtracting features for {name}...")
    feat_list, pred_list = [], []
    for imgs_01, labs in raw_test_ds:
        preprocessed = preprocess_for_model(imgs_01, name)
        feats = feature_extractors[name](preprocessed, training=False)
        logits = models[name](preprocessed, training=False)
        preds = tf.argmax(logits, axis=-1)
        feat_list.append(feats.numpy())
        pred_list.append(preds.numpy())

    test_features[name] = np.concatenate(feat_list, axis=0)
    test_preds[name] = np.concatenate(pred_list, axis=0)
    correct_masks[name] = (test_preds[name] == test_labels)

    n_correct = correct_masks[name].sum()
    print(f"  {name}: {n_correct}/{len(test_labels)} correctly classified "
          f"({n_correct/len(test_labels)*100:.1f}%)")
    print(f"  Feature shape: {test_features[name].shape}")

print(f"\n\u2705 Feature extraction complete for all models.")

## 9. Compute Class Prototypes

For each model, compute the mean feature vector per class using only correctly classified test samples.

$\mu_c = \frac{1}{|S_c|} \sum_{x \in S_c} f(x)$

In [ ]:
class_prototypes: Dict[str, np.ndarray] = {}  # {model_name: (num_classes, feat_dim)}

for name in MODEL_NAMES:
    feat_dim = test_features[name].shape[1]
    prototypes = np.zeros((NUM_CLASSES, feat_dim), dtype=np.float32)
    counts = np.zeros(NUM_CLASSES, dtype=np.int32)

    mask = correct_masks[name]
    for i in range(len(test_labels)):
        if mask[i]:
            c = test_labels[i]
            prototypes[c] += test_features[name][i]
            counts[c] += 1

    # Avoid division by zero for classes with no correct samples
    safe_counts = np.maximum(counts, 1)
    prototypes = prototypes / safe_counts[:, None]
    class_prototypes[name] = prototypes

    non_empty = (counts > 0).sum()
    print(f"{name}: prototypes computed for {non_empty}/{NUM_CLASSES} classes "
          f"(feat_dim={feat_dim})")

print(f"\n\u2705 Class prototypes computed for all models.")

## 10. Feature Inversion Utility

Given a target feature vector, optimise a [0,1] image to match that feature representation.  
Used by the Latent Interpolation and Prototype Shift attacks.

$\hat{x} = \arg\min_{x \in [0,1]} \| f(x) - z_{target} \|_2^2 + \lambda_{TV} \cdot TV(x)$

In [ ]:
class _FeatureInverter:
    """Compile the gradient step once per model with @tf.function.

    Avoids re-tracing on every call (900+ calls during prototype shift).
    Speedup vs eager: 5-10x on CPU, 2-4x on GPU.
    """

    def __init__(self):
        self._cache: dict = {}

    def _setup(self, model_name: str, lr: float, tv_weight: float) -> None:
        feat_model = feature_extractors[model_name]
        img_var = tf.Variable(
            tf.zeros([1, IMG_SIZE[0], IMG_SIZE[1], 3], dtype=tf.float32),
            trainable=True, name=f"inv_img_{model_name}"
        )
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
        tv_w = tf.constant(tv_weight, dtype=tf.float32)

        # @tf.function traces once per model_name (closure captures model_name as
        # a Python string, so TF creates one concrete function per model).
        @tf.function
        def _step(target_t: tf.Tensor) -> tf.Tensor:
            with tf.GradientTape() as tape:
                preprocessed = preprocess_for_model(img_var, model_name)
                feats = feat_model(preprocessed, training=False)
                feat_loss = tf.reduce_mean(tf.square(feats - target_t))
                tv_loss = tf.reduce_sum(tf.image.total_variation(img_var))
                loss = feat_loss + tv_w * tv_loss
            grads = tape.gradient(loss, [img_var])
            optimizer.apply_gradients(zip(grads, [img_var]))
            img_var.assign(tf.clip_by_value(img_var, 0.0, 1.0))
            return loss

        self._cache[model_name] = {
            'img_var': img_var,
            'step': _step,
        }

    def invert(self, target_features: np.ndarray, init_image_01: np.ndarray,
               model_name: str, steps: int, lr: float,
               tv_weight: float) -> np.ndarray:
        if model_name not in self._cache:
            self._setup(model_name, lr, tv_weight)

        cache = self._cache[model_name]
        img_var: tf.Variable = cache['img_var']
        _step = cache['step']

        img_var.assign(init_image_01[None, ...])
        target_t = tf.constant(target_features[None, :], dtype=tf.float32)

        prev_loss = float('inf')
        for _ in range(steps):
            loss = float(_step(target_t))
            # Early stop: converged or loss below threshold
            if loss < 1e-3 or (prev_loss - loss) < 1e-6 * max(prev_loss, 1e-8):
                break
            prev_loss = loss

        return img_var.numpy()[0]


_inverter = _FeatureInverter()


def invert_features(target_features: np.ndarray,
                    init_image_01: np.ndarray,
                    model_name: str,
                    steps: int = INVERSION_STEPS,
                    lr: float = INVERSION_LR,
                    tv_weight: float = 1e-4) -> np.ndarray:
    """Optimise image pixels to match target feature vector.

    Args:
        target_features: (feat_dim,) target feature vector.
        init_image_01: (H, W, 3) initial image in [0,1].
        model_name: which model's features to match.
        steps: max optimisation steps (early-stopped on convergence).
        lr: learning rate.
        tv_weight: total variation regularisation weight.

    Returns:
        Optimised image in [0,1] as numpy array (H, W, 3).
    """
    return _inverter.invert(target_features, init_image_01, model_name,
                            steps, lr, tv_weight)

print("\u2705 Feature inversion utility defined (cached @tf.function, early stopping).")


## 10b. Feature Inversion Quality Validation

Before using inversion-dependent attacks, validate that feature inversion
faithfully reconstructs images. Key metric: **classification preservation rate**
(does the model still predict the same class after round-trip inversion?).
Any fooling rate below the inversion-only baseline is not attributable to the attack.

In [ ]:
def evaluate_inversion_quality(model_name: str, n_samples: int = 50) -> dict:
    """Validate feature inversion by round-tripping: image -> features -> inversion -> classify.

    Returns dict with SSIM, L2, feature error, and classification preservation rate.
    """
    mask = correct_masks[model_name]
    correct_idx = np.where(mask)[0]
    np.random.seed(SEED)
    sample_idx = np.random.choice(correct_idx, min(n_samples, len(correct_idx)), replace=False)

    ssims, l2_dists, feat_errors, preserved = [], [], [], []
    example_pairs = []  # store a few for visualization

    for count, idx in enumerate(sample_idx):
        orig_img = load_single_image(test_paths[idx])
        orig_feat = test_features[model_name][idx]

        # Round-trip: invert original features back to image
        recon_img = invert_features(orig_feat, orig_img, model_name)

        # SSIM (expects 4-D tensors)
        ssim_val = float(tf.image.ssim(
            tf.constant(orig_img[None, ...], dtype=tf.float32),
            tf.constant(recon_img[None, ...], dtype=tf.float32),
            max_val=1.0
        )[0])
        ssims.append(ssim_val)

        # L2 pixel distance (normalized)
        l2 = float(np.sqrt(np.mean((orig_img - recon_img) ** 2)))
        l2_dists.append(l2)

        # Feature reconstruction error
        recon_feat = feature_extractors[model_name](
            preprocess_for_model(tf.constant(recon_img[None, ...], tf.float32), model_name),
            training=False
        ).numpy()[0]
        feat_err = float(np.linalg.norm(orig_feat - recon_feat) / (np.linalg.norm(orig_feat) + 1e-8))
        feat_errors.append(feat_err)

        # Classification preservation
        pred, conf = predict_single(recon_img, model_name)
        true_label = int(test_labels[idx])
        preserved.append(pred == true_label)

        if len(example_pairs) < 2:
            example_pairs.append({
                'original': orig_img, 'reconstructed': recon_img,
                'ssim': ssim_val, 'true': true_label, 'pred': pred
            })

        if count % 10 == 0:
            print(f"  {count}/{len(sample_idx)}...", end="\r")

    preservation_rate = float(np.mean(preserved))
    inversion_baseline_fooling = 1.0 - preservation_rate

    results = {
        'mean_ssim': float(np.mean(ssims)),
        'std_ssim': float(np.std(ssims)),
        'mean_l2': float(np.mean(l2_dists)),
        'mean_feat_error': float(np.mean(feat_errors)),
        'classification_preservation_rate': preservation_rate,
        'inversion_baseline_fooling': inversion_baseline_fooling,
        'n_samples': len(sample_idx),
    }
    return results, example_pairs

print("\u2705 evaluate_inversion_quality() defined.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# RUN INVERSION QUALITY VALIDATION
# ════════════════════════════════════════════════════════════════

inversion_quality: Dict[str, dict] = {}
inversion_examples: Dict[str, list] = {}

for name in MODEL_NAMES:
    print(f"\nValidating inversion for {name}...")
    results, examples = evaluate_inversion_quality(name, n_samples=50)
    inversion_quality[name] = results
    inversion_examples[name] = examples

    print(f"  SSIM:          {results['mean_ssim']:.3f} +/- {results['std_ssim']:.3f}")
    print(f"  L2 pixel:      {results['mean_l2']:.4f}")
    print(f"  Feature error: {results['mean_feat_error']:.4f}")
    print(f"  Class preserved: {results['classification_preservation_rate']*100:.1f}%")
    print(f"  Inversion-only baseline fooling: {results['inversion_baseline_fooling']*100:.1f}%")

    if results['classification_preservation_rate'] < 0.5:
        print(f"  \u26a0\ufe0f  WARNING: Low preservation rate for {name} — "
              "inversion-dependent attack results should be interpreted with caution.")

# Save
with open(RESULTS_DIR / 'inversion_quality.json', 'w') as f:
    json.dump(inversion_quality, f, indent=2)

# Visualize example pairs (2 per model)
fig, axes = plt.subplots(len(MODEL_NAMES), 4, figsize=(14, 3 * len(MODEL_NAMES)))
for row, name in enumerate(MODEL_NAMES):
    for col_pair in range(2):
        ex = inversion_examples[name][col_pair]
        ax_orig = axes[row, col_pair * 2]
        ax_recon = axes[row, col_pair * 2 + 1]
        ax_orig.imshow(np.clip(ex['original'], 0, 1))
        ax_orig.set_title(f"{name}\nOriginal (class {ex['true']})", fontsize=9)
        ax_orig.axis('off')
        ax_recon.imshow(np.clip(ex['reconstructed'], 0, 1))
        ax_recon.set_title(f"Reconstructed\nSSIM={ex['ssim']:.3f}, pred={ex['pred']}", fontsize=9)
        ax_recon.axis('off')

plt.suptitle("Feature Inversion Quality: Original vs Reconstructed", fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'inversion_quality_grid.png', dpi=200)
plt.show()

# Print summary table
print("\n" + "=" * 70)
print(f"{'Model':<15} {'SSIM':>8} {'L2':>8} {'Feat Err':>10} {'Preserved':>12} {'Inv Baseline':>14}")
print("-" * 70)
for name in MODEL_NAMES:
    r = inversion_quality[name]
    print(f"{name:<15} {r['mean_ssim']:>8.3f} {r['mean_l2']:>8.4f} "
          f"{r['mean_feat_error']:>10.4f} {r['classification_preservation_rate']*100:>11.1f}% "
          f"{r['inversion_baseline_fooling']*100:>13.1f}%")
print("=" * 70)

print("\n\u2705 Inversion quality validated and saved.")


---
# PART 2 — ATTACK IMPLEMENTATIONS

---

## 11. Attack 1 — Latent Interpolation Attack

**Idea:** Simulate representation drift by interpolating in latent space.

For each correctly classified sample $x$ with label $A$:
1. Find its nearest neighbour $x_{target}$ from a different class $B$ in feature space.
2. Interpolate: $z = (1-\alpha) f(x) + \alpha f(x_{target})$ for $\alpha \in \{0.1, 0.2, 0.3, 0.4\}$.
3. Reconstruct image via feature inversion.
4. Measure classification change vs $\alpha$.

In [ ]:
def find_nearest_different_class(features: np.ndarray, labels: np.ndarray,
                                  query_idx: int, mask: np.ndarray) -> int:
    """Vectorized nearest-neighbour from a different class."""
    query_feat = features[query_idx]
    query_label = labels[query_idx]

    # Boolean mask: different class AND correctly classified AND not self
    valid = mask.copy()
    valid[query_idx] = False
    valid &= (labels != query_label)

    if not valid.any():
        return -1

    dists = np.sum((features[valid] - query_feat) ** 2, axis=1)
    return int(np.where(valid)[0][np.argmin(dists)])


# ── Image cache (avoids re-reading the same file from disk) ──
_image_cache: Dict[str, np.ndarray] = {}

def load_single_image(path: str, img_size: Tuple[int, int] = IMG_SIZE) -> np.ndarray:
    """Load a single image as [0,1] numpy array (H, W, 3). Cached."""
    if path not in _image_cache:
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        _image_cache[path] = (tf.cast(img, tf.float32) / 255.0).numpy()
    return _image_cache[path]


def predict_single(image_01: np.ndarray, model_name: str) -> Tuple[int, float]:
    """Predict class and confidence for a single [0,1] image."""
    preds, confs = predict_batch(image_01[None, ...], model_name)
    return int(preds[0]), float(confs[0])


def predict_batch(images_01: np.ndarray, model_name: str) -> Tuple[np.ndarray, np.ndarray]:
    """Predict classes and confidences for a batch of [0,1] images.

    Args:
        images_01: (N, H, W, 3) array in [0,1].
    Returns:
        preds (N,) int, confs (N,) float.
    """
    preprocessed = preprocess_for_model(
        tf.constant(images_01, dtype=tf.float32), model_name
    )
    logits = models[model_name](preprocessed, training=False)
    probs = tf.nn.softmax(logits, axis=-1).numpy()
    preds = np.argmax(probs, axis=1)
    confs = probs[np.arange(len(preds)), preds]
    return preds, confs

print("\u2705 Helper functions defined (vectorized NN, batch predict, image cache).")


In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK 1: LATENT INTERPOLATION ATTACK
# ════════════════════════════════════════════════════════════════

MAX_SAMPLES_PER_ATTACK = 100  # Cap for computational tractability

latent_interp_results: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Latent Interpolation Attack — {name}")
    print(f"{'='*60}")

    mask = correct_masks[name]
    correct_indices = np.where(mask)[0]
    np.random.seed(SEED)
    if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
        sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
    else:
        sample_indices = correct_indices
    sample_indices = np.sort(sample_indices)

    alpha_results = {alpha: {'fooled': 0, 'total': 0,
                             'class_changes': []} for alpha in INTERP_ALPHAS}

    for count, idx in enumerate(sample_indices):
        if count % 20 == 0:
            print(f"  Processing {count}/{len(sample_indices)}...")

        nn_idx = find_nearest_different_class(
            test_features[name], test_labels, idx, mask
        )
        if nn_idx == -1:
            continue

        orig_feat = test_features[name][idx]
        target_feat = test_features[name][nn_idx]
        true_label = int(test_labels[idx])
        target_label = int(test_labels[nn_idx])
        orig_image = load_single_image(test_paths[idx])

        # Invert all alphas, then batch-predict
        recons = []
        for alpha in INTERP_ALPHAS:
            z_interp = (1 - alpha) * orig_feat + alpha * target_feat
            recons.append(invert_features(z_interp, orig_image, name))

        preds, confs = predict_batch(np.array(recons), name)

        for ai, alpha in enumerate(INTERP_ALPHAS):
            pred, conf = int(preds[ai]), float(confs[ai])
            alpha_results[alpha]['total'] += 1
            if pred != true_label:
                alpha_results[alpha]['fooled'] += 1
            alpha_results[alpha]['class_changes'].append({
                'idx': int(idx),
                'true': true_label,
                'target': target_label,
                'pred': pred,
                'conf': conf,
                'fooled': pred != true_label
            })

    # Summarise
    model_summary = {}
    for alpha in INTERP_ALPHAS:
        r = alpha_results[alpha]
        rate = r['fooled'] / r['total'] if r['total'] > 0 else 0.0
        model_summary[str(alpha)] = {
            'fooling_rate': rate,
            'fooled': r['fooled'],
            'total': r['total'],
            'class_changes': r['class_changes']
        }
        print(f"  alpha={alpha}: fooling rate = {rate*100:.1f}% "
              f"({r['fooled']}/{r['total']})")

    latent_interp_results[name] = model_summary

print(f"\n\u2705 Latent Interpolation Attack complete.")


## 12. Attack 2 — Feature Distribution Matching Attack

**Hypothesis:** Aligning a sample's feature distribution statistics (mean, variance) to match a different-class sample may cause misclassification.

**Method:** Adaptive Instance Normalisation (AdaIN) applied to the penultimate-layer feature vector (global average pooling output). Unlike spatial AdaIN which transfers textures via intermediate convolutional maps, this operates on the 1-D global feature vector, testing feature distribution sensitivity rather than texture bias:

$$\text{AdaIN}(x, y) = \sigma(y) \left( \frac{x - \mu(x)}{\sigma(x)} \right) + \mu(y)$$

Content = original image, Style = random image from a different class.

In [ ]:
def adain_style_transfer(content: np.ndarray, style: np.ndarray,
                          model_name: str, alpha: float = 1.0) -> np.ndarray:
    """Apply feature distribution matching via AdaIN, then invert.

    Aligns the feature distribution statistics (mean, variance) of the content
    sample to match those of a different-class style sample. Operates on the
    penultimate-layer global feature vector (1-D), not spatial feature maps.

    Tests whether feature distribution alignment alone can induce misclassification.
    """
    # Use the feature extractor (penultimate layer) for AdaIN
    content_t = tf.constant(content[None, ...], dtype=tf.float32)
    style_t = tf.constant(style[None, ...], dtype=tf.float32)

    content_pre = preprocess_for_model(content_t, model_name)
    style_pre = preprocess_for_model(style_t, model_name)

    content_feat = feature_extractors[model_name](content_pre, training=False).numpy()[0]
    style_feat = feature_extractors[model_name](style_pre, training=False).numpy()[0]

    # AdaIN in feature space (1-D vector: channel-level stats)
    c_mean, c_std = content_feat.mean(), content_feat.std() + 1e-8
    s_mean, s_std = style_feat.mean(), style_feat.std() + 1e-8

    adain_feat = s_std * (content_feat - c_mean) / c_std + s_mean
    # Blend with content features
    blended_feat = alpha * adain_feat + (1 - alpha) * content_feat

    # Invert blended features back to pixel space
    stylized = invert_features(blended_feat, content, model_name,
                               steps=75, lr=0.01)
    return stylized

print("\u2705 AdaIN style transfer function defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK 2: STYLE TRANSFER ATTACK
# ════════════════════════════════════════════════════════════════

style_transfer_results: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Style Transfer Attack — {name}")
    print(f"{'='*60}")

    mask = correct_masks[name]
    correct_indices = np.where(mask)[0]
    np.random.seed(SEED)
    if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
        sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
    else:
        sample_indices = correct_indices
    sample_indices = np.sort(sample_indices)

    fooled_count = 0
    total_count = 0
    class_pair_fooling = defaultdict(lambda: {'fooled': 0, 'total': 0})
    details = []

    for count, idx in enumerate(sample_indices):
        if count % 20 == 0:
            print(f"  Processing {count}/{len(sample_indices)}...")

        true_label = int(test_labels[idx])
        content_img = load_single_image(test_paths[idx])

        diff_class_indices = np.where(
            (test_labels != true_label) & mask
        )[0]
        if len(diff_class_indices) == 0:
            continue
        style_idx = np.random.choice(diff_class_indices)
        style_label = int(test_labels[style_idx])
        style_img = load_single_image(test_paths[style_idx])

        stylized = adain_style_transfer(content_img, style_img, name)
        pred, conf = predict_single(stylized, name)

        total_count += 1
        is_fooled = pred != true_label
        if is_fooled:
            fooled_count += 1

        pair_key = f"{idx_to_class[true_label]}->{idx_to_class[style_label]}"
        class_pair_fooling[pair_key]['total'] += 1
        if is_fooled:
            class_pair_fooling[pair_key]['fooled'] += 1

        details.append({
            'idx': int(idx),
            'true': true_label,
            'style_class': style_label,
            'pred': pred,
            'conf': conf,
            'fooled': is_fooled
        })

    rate = fooled_count / total_count if total_count > 0 else 0.0
    style_transfer_results[name] = {
        'fooling_rate': rate,
        'fooled': fooled_count,
        'total': total_count,
        'class_pair_fooling': dict(class_pair_fooling),
        'details': details
    }
    print(f"  Fooling rate: {rate*100:.1f}% ({fooled_count}/{total_count})")

print(f"\n\u2705 Style Transfer Attack complete.")


## 13. Attack 3 — Prototype Shift Attack

For sample $x$ from class $A$, shift its feature representation toward class $B$'s prototype:

$$f_{adv} = f(x) + \lambda (\mu_B - \mu_A)$$

$\lambda \in \{0.2, 0.4, 0.6\}$

Reconstruct image and measure the minimum $\lambda$ that causes label change.

In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK 3: PROTOTYPE SHIFT ATTACK
# ════════════════════════════════════════════════════════════════

proto_shift_results: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Prototype Shift Attack — {name}")
    print(f"{'='*60}")

    mask = correct_masks[name]
    correct_indices = np.where(mask)[0]
    np.random.seed(SEED)
    if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
        sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
    else:
        sample_indices = correct_indices
    sample_indices = np.sort(sample_indices)

    lambda_results = {lam: {'fooled': 0, 'total': 0,
                            'details': []} for lam in PROTO_LAMBDAS}
    min_lambda_for_change = []

    for count, idx in enumerate(sample_indices):
        if count % 20 == 0:
            print(f"  Processing {count}/{len(sample_indices)}...")

        true_label = int(test_labels[idx])
        orig_feat = test_features[name][idx]
        orig_img = load_single_image(test_paths[idx])

        target_classes = [c for c in range(NUM_CLASSES) if c != true_label]
        target_class = np.random.choice(target_classes)

        proto_A = class_prototypes[name][true_label]
        proto_B = class_prototypes[name][target_class]
        shift_dir = proto_B - proto_A

        # Invert all lambdas, then batch-predict
        recons = []
        for lam in PROTO_LAMBDAS:
            f_adv = orig_feat + lam * shift_dir
            recons.append(invert_features(f_adv, orig_img, name))

        preds, confs = predict_batch(np.array(recons), name)

        sample_min_lambda = None
        for li, lam in enumerate(PROTO_LAMBDAS):
            pred, conf = int(preds[li]), float(confs[li])
            lambda_results[lam]['total'] += 1
            is_fooled = pred != true_label
            if is_fooled:
                lambda_results[lam]['fooled'] += 1
                if sample_min_lambda is None:
                    sample_min_lambda = lam

            lambda_results[lam]['details'].append({
                'idx': int(idx),
                'true': true_label,
                'target_class': int(target_class),
                'pred': pred,
                'conf': conf,
                'fooled': is_fooled
            })

        if sample_min_lambda is not None:
            min_lambda_for_change.append(sample_min_lambda)

    # Summarise
    model_summary = {}
    for lam in PROTO_LAMBDAS:
        r = lambda_results[lam]
        rate = r['fooled'] / r['total'] if r['total'] > 0 else 0.0
        model_summary[str(lam)] = {
            'fooling_rate': rate,
            'fooled': r['fooled'],
            'total': r['total'],
            'details': r['details']
        }
        print(f"  lambda={lam}: fooling rate = {rate*100:.1f}% "
              f"({r['fooled']}/{r['total']})")

    avg_min_lambda = np.mean(min_lambda_for_change) if min_lambda_for_change else None
    model_summary['min_lambda_for_change'] = {
        'mean': float(avg_min_lambda) if avg_min_lambda else None,
        'values': min_lambda_for_change,
        'count': len(min_lambda_for_change)
    }
    if avg_min_lambda:
        print(f"  Mean minimum lambda for label change: {avg_min_lambda:.3f} "
              f"(n={len(min_lambda_for_change)})")

    proto_shift_results[name] = model_summary

print(f"\n\u2705 Prototype Shift Attack complete.")


### Same-Class Control (Prototype Shift)

Shift features toward the sample's **own-class** prototype with the same $\lambda$ values.
If this also fools the model, then fooling is caused by inversion noise, not the semantic direction.

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONTROL: SAME-CLASS PROTOTYPE SHIFT
# ════════════════════════════════════════════════════════════════

proto_shift_control: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\nSame-Class Control — {name}")

    mask = correct_masks[name]
    correct_indices = np.where(mask)[0]
    np.random.seed(SEED)
    if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
        sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
    else:
        sample_indices = correct_indices
    sample_indices = np.sort(sample_indices)

    control_results = {lam: {'fooled': 0, 'total': 0} for lam in PROTO_LAMBDAS}

    for count, idx in enumerate(sample_indices):
        if count % 20 == 0:
            print(f"  Processing {count}/{len(sample_indices)}...", end="\r")

        true_label = int(test_labels[idx])
        orig_feat = test_features[name][idx]
        orig_img = load_single_image(test_paths[idx])

        # Shift toward OWN class prototype (should NOT fool)
        proto_own = class_prototypes[name][true_label]
        shift_dir = proto_own - orig_feat  # toward own centroid

        recons = []
        for lam in PROTO_LAMBDAS:
            f_shifted = orig_feat + lam * shift_dir
            recons.append(invert_features(f_shifted, orig_img, name))

        preds, confs = predict_batch(np.array(recons), name)

        for li, lam in enumerate(PROTO_LAMBDAS):
            control_results[lam]['total'] += 1
            if int(preds[li]) != true_label:
                control_results[lam]['fooled'] += 1

    model_control = {}
    for lam in PROTO_LAMBDAS:
        r = control_results[lam]
        rate = r['fooled'] / r['total'] if r['total'] > 0 else 0.0
        model_control[str(lam)] = {'fooling_rate': rate, 'fooled': r['fooled'], 'total': r['total']}

    proto_shift_control[name] = model_control

    # Print comparison
    print(f"  {'Lambda':<10} {'Same-Class':>12} {'Diff-Class':>12} {'Net Effect':>12}")
    for lam in PROTO_LAMBDAS:
        same = model_control[str(lam)]['fooling_rate']
        diff = proto_shift_results[name][str(lam)]['fooling_rate']
        net = diff - same
        print(f"  {lam:<10.1f} {same*100:>11.1f}% {diff*100:>11.1f}% {net*100:>11.1f}%")

print(f"\n\u2705 Same-class control complete.")


## 14. Attack 4 — Diffusion-like Noise Attack (Semantic Denoise)

Iteratively:
1. Add Gaussian noise: $x_{noisy} = x + \mathcal{N}(0, \sigma^2)$
2. Denoise using median filter + Gaussian blur

This simulates generative prior projection.  
**Goal:** See if model predictions change while the image remains recognisable to humans.

In [ ]:
def diffusion_denoise_attack(image_01: np.ndarray,
                              steps: int = DIFFUSION_STEPS,
                              sigma: float = DIFFUSION_SIGMA,
                              median_size: int = 3,
                              blur_sigma: float = 1.0) -> List[np.ndarray]:
    """Apply iterative noise-denoise cycles.

    Returns list of images at each step (including original at step 0).
    """
    trajectory = [image_01.copy()]
    x = image_01.copy()
    for _ in range(steps):
        # Add Gaussian noise
        noise = np.random.randn(*x.shape).astype(np.float32) * sigma
        x = x + noise
        x = np.clip(x, 0.0, 1.0)
        # Denoise: median filter per channel + Gaussian blur
        for ch in range(3):
            x[:, :, ch] = median_filter(x[:, :, ch], size=median_size)
        x = gaussian_filter(x, sigma=blur_sigma)
        x = np.clip(x, 0.0, 1.0).astype(np.float32)
        trajectory.append(x.copy())
    return trajectory

print("\u2705 Diffusion-like noise attack function defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK 4: DIFFUSION-LIKE NOISE ATTACK
# ════════════════════════════════════════════════════════════════

diffusion_results: Dict[str, dict] = {}

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Diffusion-like Noise Attack — {name}")
    print(f"{'='*60}")

    mask = correct_masks[name]
    correct_indices = np.where(mask)[0]
    np.random.seed(SEED)
    if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
        sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
    else:
        sample_indices = correct_indices
    sample_indices = np.sort(sample_indices)

    step_results = {step: {'fooled': 0, 'total': 0} for step in range(DIFFUSION_STEPS + 1)}
    details = []

    for count, idx in enumerate(sample_indices):
        if count % 20 == 0:
            print(f"  Processing {count}/{len(sample_indices)}...")

        true_label = int(test_labels[idx])
        orig_img = load_single_image(test_paths[idx])

        trajectory = diffusion_denoise_attack(orig_img)

        # Batch-predict entire trajectory at once (6 images)
        preds, confs = predict_batch(np.array(trajectory), name)

        sample_detail = {'idx': int(idx), 'true': true_label, 'steps': []}
        for step_i in range(len(trajectory)):
            pred, conf = int(preds[step_i]), float(confs[step_i])
            is_fooled = pred != true_label
            step_results[step_i]['total'] += 1
            if is_fooled:
                step_results[step_i]['fooled'] += 1
            sample_detail['steps'].append({
                'step': step_i,
                'pred': pred,
                'conf': conf,
                'fooled': is_fooled
            })
        details.append(sample_detail)

    # Summarise
    model_summary = {'per_step': {}, 'details': details}
    for step in range(DIFFUSION_STEPS + 1):
        r = step_results[step]
        rate = r['fooled'] / r['total'] if r['total'] > 0 else 0.0
        model_summary['per_step'][step] = {
            'fooling_rate': rate,
            'fooled': r['fooled'],
            'total': r['total']
        }
        label = "(clean baseline)" if step == 0 else ""
        print(f"  Step {step}: fooling rate = {rate*100:.1f}% "
              f"({r['fooled']}/{r['total']}) {label}")

    diffusion_results[name] = model_summary

print(f"\n\u2705 Diffusion-like Noise Attack complete.")


### Noise-Denoise Sensitivity Analysis

Vary $\sigma \in \{0.02, 0.05, 0.1\}$ to show the attack is not dependent on a single arbitrary hyperparameter choice.

In [ ]:
# ════════════════════════════════════════════════════════════════
# SENSITIVITY ANALYSIS: sigma values for noise-denoise
# ════════════════════════════════════════════════════════════════

SENSITIVITY_SIGMAS = [0.02, 0.05, 0.1]
sensitivity_model = 'VGG19'  # run on one model for efficiency

print(f"Noise-Denoise Sensitivity Analysis on {sensitivity_model}")
print(f"Sigma values: {SENSITIVITY_SIGMAS}")

mask = correct_masks[sensitivity_model]
correct_indices = np.where(mask)[0]
np.random.seed(SEED)
if len(correct_indices) > MAX_SAMPLES_PER_ATTACK:
    sample_indices = np.random.choice(correct_indices, MAX_SAMPLES_PER_ATTACK, replace=False)
else:
    sample_indices = correct_indices
sample_indices = np.sort(sample_indices)

sigma_sensitivity: Dict[float, dict] = {}

for sigma in SENSITIVITY_SIGMAS:
    step_results = {s: {'fooled': 0, 'total': 0} for s in range(DIFFUSION_STEPS + 1)}

    for count, idx in enumerate(sample_indices):
        true_label = int(test_labels[idx])
        orig_img = load_single_image(test_paths[idx])
        trajectory = diffusion_denoise_attack(orig_img, sigma=sigma)
        preds, confs = predict_batch(np.array(trajectory), sensitivity_model)

        for step_i in range(len(trajectory)):
            step_results[step_i]['total'] += 1
            if int(preds[step_i]) != true_label:
                step_results[step_i]['fooled'] += 1

    per_step = {}
    for s in range(DIFFUSION_STEPS + 1):
        r = step_results[s]
        per_step[s] = r['fooled'] / r['total'] if r['total'] > 0 else 0.0

    sigma_sensitivity[sigma] = per_step
    final_rate = per_step[DIFFUSION_STEPS]
    print(f"  sigma={sigma:.2f}: final step fooling = {final_rate*100:.1f}%")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
for sigma in SENSITIVITY_SIGMAS:
    rates = [sigma_sensitivity[sigma][s] * 100 for s in range(DIFFUSION_STEPS + 1)]
    ax.plot(range(DIFFUSION_STEPS + 1), rates, marker='o', label=f'$\sigma$={sigma}', linewidth=2)

ax.set_xlabel('Noise-Denoise Step', fontsize=12)
ax.set_ylabel('Fooling Rate (%)', fontsize=12)
ax.set_title(f'Noise-Denoise Sensitivity to $\sigma$ ({sensitivity_model})', fontsize=14)
ax.legend()
ax.set_ylim(0, None)
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'noise_denoise_sigma_sensitivity.png', dpi=200)
plt.show()

# Save
with open(RESULTS_DIR / 'noise_denoise_sensitivity.json', 'w') as f:
    json.dump({str(k): {str(s): v for s, v in d.items()} for k, d in sigma_sensitivity.items()}, f, indent=2)

print("\u2705 Sensitivity analysis saved.")


---

# PART 3 — METRICS

---

## 15. Compute Aggregate Metrics

For each model, compute:
1. **Semantic Fooling Rate** — overall fraction of semantic attacks that cause misclassification.
2. **Minimum Semantic Distance** — minimum perturbation strength (α, λ, step) to flip the label.
3. **Class-to-Class Confusion Graph** — which class pairs are most confused under semantic attack.
4. **Texture Sensitivity Score** — fooling rate of the style transfer attack.
5. **Feature Drift Required for Misclassification** — from the prototype shift attack.

In [ ]:
# ════════════════════════════════════════════════════════════════
# AGGREGATE METRICS (with bootstrap CIs and inversion baseline)
# ════════════════════════════════════════════════════════════════

def bootstrap_ci(fooled_array: np.ndarray, n_bootstrap: int = 10000,
                 ci: float = 0.95) -> Tuple[float, float]:
    """Compute bootstrap confidence interval for a binary fooled array."""
    rng = np.random.RandomState(SEED)
    n = len(fooled_array)
    boot_rates = np.array([
        rng.choice(fooled_array, size=n, replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    alpha = (1 - ci) / 2
    return float(np.percentile(boot_rates, alpha * 100)), float(np.percentile(boot_rates, (1 - alpha) * 100))


aggregate_metrics: Dict[str, dict] = {}
bootstrap_cis: Dict[str, dict] = {}   # model -> attack -> (lo, hi)
confusion_matrices: Dict[str, np.ndarray] = {}  # model -> (NUM_CLASSES, NUM_CLASSES)

for name in MODEL_NAMES:
    print(f"\n{'='*60}")
    print(f"Metrics — {name}")
    print(f"{'='*60}")

    inv_baseline = inversion_quality[name]['inversion_baseline_fooling']
    print(f"  Inversion-only baseline: {inv_baseline*100:.1f}%")

    # ── Per-attack fooling rates with CIs ──
    best_alpha = str(max(INTERP_ALPHAS))
    best_lam = str(max(PROTO_LAMBDAS))

    # Build boolean arrays for bootstrap
    li_fooled = np.array([e['fooled'] for e in latent_interp_results[name][best_alpha]['class_changes']])
    st_fooled = np.array([e['fooled'] for e in style_transfer_results[name]['details']])
    ps_fooled = np.array([e['fooled'] for e in proto_shift_results[name][best_lam]['details']])
    dn_fooled = np.array([s['steps'][-1]['fooled'] for s in diffusion_results[name]['details']])

    attack_data = {
        'latent_interpolation': (li_fooled, best_alpha),
        'feature_distribution': (st_fooled, None),
        'prototype_shift': (ps_fooled, best_lam),
        'noise_denoise': (dn_fooled, None),
    }

    per_attack_rates = {}
    model_cis = {}

    for attack_key, (fooled_arr, param) in attack_data.items():
        rate = float(fooled_arr.mean())
        ci_lo, ci_hi = bootstrap_ci(fooled_arr.astype(float))
        net_rate = max(0.0, rate - inv_baseline)
        per_attack_rates[attack_key] = rate
        model_cis[attack_key] = {'rate': rate, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'net_rate': net_rate}

        param_str = f" ({param})" if param else ""
        print(f"  {attack_key}{param_str}: {rate*100:.1f}% [{ci_lo*100:.1f}, {ci_hi*100:.1f}] "
              f"(net: {net_rate*100:.1f}%)")

    bootstrap_cis[name] = model_cis

    # ── Minimum semantic distance ──
    min_alpha = None
    for a in INTERP_ALPHAS:
        if latent_interp_results[name][str(a)]['fooling_rate'] > 0:
            min_alpha = a
            break

    proto_min_lam = proto_shift_results[name].get('min_lambda_for_change', {})
    mean_min_lambda = proto_min_lam.get('mean', None)

    # ── Confusion matrix ──
    confusion_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int32)

    for entry in latent_interp_results[name][best_alpha]['class_changes']:
        if entry['fooled']:
            confusion_matrix[entry['true'], entry['pred']] += 1

    for entry in style_transfer_results[name]['details']:
        if entry['fooled']:
            confusion_matrix[entry['true'], entry['pred']] += 1

    for entry in proto_shift_results[name][best_lam]['details']:
        if entry['fooled']:
            confusion_matrix[entry['true'], entry['pred']] += 1

    for sample in diffusion_results[name]['details']:
        final = sample['steps'][-1]
        if final['fooled']:
            confusion_matrix[sample['true'], final['pred']] += 1

    confusion_matrices[name] = confusion_matrix
    total_confusions = confusion_matrix.sum()
    print(f"  Confusion matrix: {total_confusions} total misclassifications")

    # ── Feature drift ──
    print(f"  Min alpha for any fooling: {min_alpha}")
    print(f"  Mean min lambda for flip: {mean_min_lambda}")

    aggregate_metrics[name] = {
        'per_attack_fooling_rates': per_attack_rates,
        'bootstrap_cis': model_cis,
        'inversion_baseline_fooling': inv_baseline,
        'min_semantic_distance': {
            'min_alpha_for_fooling': min_alpha,
            'mean_min_lambda_for_flip': mean_min_lambda
        },
        'confusion_matrix_total': int(total_confusions),
        'distribution_sensitivity_score': per_attack_rates['feature_distribution'],
        'feature_drift_for_misclassification': mean_min_lambda,
    }

    np.save(RESULTS_DIR / f'{name}_semantic_confusion_matrix.npy', confusion_matrix)

print(f"\n\u2705 All metrics computed with bootstrap CIs.")


## 15b. Semantic Structure Score (SSS) — GenAI Attacks

Compute the Semantic Structure Score for generative attacks and compare with gradient attacks.

$$\text{SSS} = 1 - \frac{H(\text{off-diagonal})}{H_{\max}}$$

- **SSS near 1**: Structured failures (model confuses to few semantically related classes)
- **SSS near 0**: Random failures (model confuses to many unrelated classes)

In [ ]:
# ════════════════════════════════════════════════════════════════
# SEMANTIC STRUCTURE SCORE (SSS) FOR GENAI ATTACKS
# ════════════════════════════════════════════════════════════════
from scipy.stats import entropy as scipy_entropy


def compute_semantic_structure_score(C: np.ndarray) -> float:
    """Compute SSS from confusion matrix.
    Reused from experiments/confusion_direction_analysis.py:110-130.

    SSS = 1 - H(off_diagonal) / H_max_possible
    """
    num_classes = C.shape[0]
    mask = ~np.eye(num_classes, dtype=bool)
    off_diag = C[mask].flatten().astype(float)
    total_off = off_diag.sum()

    if total_off == 0:
        return 1.0

    probs = off_diag / total_off
    H = float(scipy_entropy(probs + 1e-12, base=2))
    H_max = np.log2(len(off_diag))
    return round(1.0 - (H / H_max) if H_max > 0 else 1.0, 6)


def compute_per_class_entropy(C: np.ndarray) -> np.ndarray:
    """Compute confusion entropy for each true class row."""
    entropies = np.zeros(C.shape[0])
    for i in range(C.shape[0]):
        row = C[i, :].copy().astype(float)
        row[i] = 0  # exclude diagonal
        total = row.sum()
        if total > 0:
            probs = row / total
            entropies[i] = float(scipy_entropy(probs + 1e-12, base=2))
        else:
            entropies[i] = np.nan  # no misclassifications for this class
    return entropies


def extract_top_confused_pairs(C: np.ndarray, class_names: list, top_k: int = 10) -> list:
    """Extract top confused (true_class, adv_class) pairs by count.
    Reused from experiments/confusion_direction_analysis.py:133-147.
    """
    pairs = []
    for i in range(C.shape[0]):
        for j in range(C.shape[1]):
            if i != j and C[i, j] > 0:
                pairs.append({
                    'true_label': i, 'true_class': class_names[i],
                    'adv_label': j, 'adv_class': class_names[j],
                    'count': int(C[i, j]),
                })
    pairs.sort(key=lambda x: x['count'], reverse=True)
    return pairs[:top_k]


# ── Compute SSS for each model ──
sss_genai: Dict[str, dict] = {}
per_class_entropies: Dict[str, np.ndarray] = {}

for name in MODEL_NAMES:
    C = confusion_matrices[name]
    sss = compute_semantic_structure_score(C)
    pce = compute_per_class_entropy(C)
    top_pairs = extract_top_confused_pairs(C, class_names)

    valid_entropies = pce[~np.isnan(pce)]
    sss_genai[name] = {
        'sss': sss,
        'mean_per_class_entropy': float(np.mean(valid_entropies)) if len(valid_entropies) > 0 else None,
        'median_per_class_entropy': float(np.median(valid_entropies)) if len(valid_entropies) > 0 else None,
        'top_confused_pairs': top_pairs,
    }
    per_class_entropies[name] = pce

    print(f"{name}: SSS = {sss:.4f} | Mean entropy = {sss_genai[name]['mean_per_class_entropy']:.3f}")
    print(f"  Top-5 confused pairs:")
    for p in top_pairs[:5]:
        print(f"    {p['true_class']} -> {p['adv_class']}: {p['count']}")

# Try loading gradient attack SSS for comparison
gradient_sss_path = Path('fgsm_results') / 'confusion_analysis'
gradient_sss = {}
for csv_f in gradient_sss_path.glob('*semantic_structure*') if gradient_sss_path.exists() else []:
    try:
        df = pd.read_csv(csv_f)
        for _, row in df.iterrows():
            gradient_sss[row.get('model', '')] = row.get('sss', None)
    except Exception:
        pass

if gradient_sss:
    print(f"\nComparison with gradient attacks:")
    print(f"  {'Model':<15} {'SSS (GenAI)':>12} {'SSS (Gradient)':>14}")
    for name in MODEL_NAMES:
        g_sss = gradient_sss.get(name, 'N/A')
        print(f"  {name:<15} {sss_genai[name]['sss']:>12.4f} {str(g_sss):>14}")

# Save
sss_rows = []
for name in MODEL_NAMES:
    sss_rows.append({
        'model': name, 'sss_genai': sss_genai[name]['sss'],
        'mean_entropy': sss_genai[name]['mean_per_class_entropy'],
        'median_entropy': sss_genai[name]['median_per_class_entropy'],
    })
pd.DataFrame(sss_rows).to_csv(RESULTS_DIR / 'genai_semantic_structure_scores.csv', index=False)

with open(RESULTS_DIR / 'genai_top_confused_pairs.json', 'w') as f:
    json.dump({name: sss_genai[name]['top_confused_pairs'] for name in MODEL_NAMES}, f, indent=2, default=str)

print(f"\n\u2705 SSS computed and saved.")


In [ ]:
# ════════════════════════════════════════════════════════════════
# SSS VISUALIZATION
# ════════════════════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: SSS bar chart
sss_vals = [sss_genai[name]['sss'] for name in MODEL_NAMES]
bar_colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax1.bar(MODEL_NAMES, sss_vals, color=bar_colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, sss_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=11)
ax1.set_ylabel('Semantic Structure Score (SSS)', fontsize=12)
ax1.set_title('SSS per Model (GenAI Attacks)', fontsize=14)
ax1.set_ylim(0, 1.0)
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Right: Per-class entropy boxplot
entropy_data = [per_class_entropies[name][~np.isnan(per_class_entropies[name])] for name in MODEL_NAMES]
bp = ax2.boxplot(entropy_data, labels=MODEL_NAMES, patch_artist=True)
for patch, color in zip(bp['boxes'], bar_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax2.set_ylabel('Per-Class Confusion Entropy (bits)', fontsize=12)
ax2.set_title('Confusion Entropy Distribution', fontsize=14)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'sss_comparison_genai.png', dpi=200)
plt.show()
print("\u2705 SSS visualization saved.")


## 16. Per-Class Vulnerability Heatmap

In [ ]:
# ════════════════════════════════════════════════════════════════
# PER-CLASS FOOLING HEATMAP
# ════════════════════════════════════════════════════════════════

heatmap_rows = []

for name in MODEL_NAMES:
    class_fooled = defaultdict(lambda: {'total': 0, 'fooled': 0})

    # Aggregate all attacks
    # Latent interpolation (strongest alpha)
    best_alpha = str(max(INTERP_ALPHAS))
    for entry in latent_interp_results[name][best_alpha]['class_changes']:
        cls = entry['true']
        class_fooled[cls]['total'] += 1
        if entry['fooled']:
            class_fooled[cls]['fooled'] += 1

    # Style transfer
    for entry in style_transfer_results[name]['details']:
        cls = entry['true']
        class_fooled[cls]['total'] += 1
        if entry['fooled']:
            class_fooled[cls]['fooled'] += 1

    # Prototype shift (strongest lambda)
    best_lam = str(max(PROTO_LAMBDAS))
    for entry in proto_shift_results[name][best_lam]['details']:
        cls = entry['true']
        class_fooled[cls]['total'] += 1
        if entry['fooled']:
            class_fooled[cls]['fooled'] += 1

    # Diffusion (final step)
    for sample in diffusion_results[name]['details']:
        cls = sample['true']
        final = sample['steps'][-1]
        class_fooled[cls]['total'] += 1
        if final['fooled']:
            class_fooled[cls]['fooled'] += 1

    for cls_idx in sorted(class_fooled.keys()):
        d = class_fooled[cls_idx]
        rate = d['fooled'] / d['total'] if d['total'] > 0 else 0.0
        heatmap_rows.append({
            'model': name,
            'class_idx': cls_idx,
            'class_name': idx_to_class[cls_idx],
            'total': d['total'],
            'fooled': d['fooled'],
            'fooling_rate': rate
        })

df_heatmap = pd.DataFrame(heatmap_rows)
heatmap_csv = RESULTS_DIR / 'genai_per_class_heatmap.csv'
df_heatmap.to_csv(heatmap_csv, index=False)

print(f"Per-class heatmap saved to {heatmap_csv}")
print(f"Shape: {df_heatmap.shape}")
print(f"\nTop 10 most vulnerable classes (across models):")
print(df_heatmap.sort_values('fooling_rate', ascending=False).head(10)[
    ['model', 'class_name', 'fooling_rate', 'fooled', 'total']
].to_string(index=False))

## 17. Visualise Per-Class Heatmap

In [ ]:
# ════════════════════════════════════════════════════════════════
# HEATMAP FIGURE — Top 20 most vulnerable classes
# ════════════════════════════════════════════════════════════════

pivot = df_heatmap.pivot_table(
    index='class_name', columns='model', values='fooling_rate'
)

# Select top 20 by mean fooling rate across models
pivot['mean_rate'] = pivot.mean(axis=1)
top_classes = pivot.nlargest(20, 'mean_rate').drop(columns='mean_rate')

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(
    top_classes,
    annot=True, fmt='.2f', cmap='YlOrRd',
    linewidths=0.5, linecolor='#333333',
    cbar_kws={'label': 'Semantic Fooling Rate'},
    ax=ax
)
ax.set_title('Top 20 Most Vulnerable Classes — Generative Attacks',
             fontsize=14, pad=15)
ax.set_ylabel('Class')
ax.set_xlabel('Model')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'per_class_vulnerability_heatmap.png', dpi=200)
plt.show()
print(f"\u2705 Heatmap saved.")

## 18. Confusion Matrix Visualisation

In [ ]:
# ════════════════════════════════════════════════════════════════
# SEMANTIC CONFUSION MATRICES
# ════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for ax, name in zip(axes, MODEL_NAMES):
    cm = np.load(RESULTS_DIR / f'{name}_semantic_confusion_matrix.npy')
    # Show only non-zero region for readability
    row_mask = cm.sum(axis=1) > 0
    col_mask = cm.sum(axis=0) > 0
    combined_mask = row_mask | col_mask
    if combined_mask.sum() > 0:
        cm_sub = cm[np.ix_(combined_mask, combined_mask)]
        sub_names = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if combined_mask[i]]
    else:
        cm_sub = cm[:20, :20]
        sub_names = CLASS_NAMES[:20]

    # Limit to max 25 for readability
    if len(sub_names) > 25:
        top_idx = np.argsort(cm_sub.sum(axis=1) + cm_sub.sum(axis=0))[-25:]
        cm_sub = cm_sub[np.ix_(top_idx, top_idx)]
        sub_names = [sub_names[i] for i in top_idx]

    sns.heatmap(
        cm_sub, xticklabels=sub_names, yticklabels=sub_names,
        cmap='Reds', linewidths=0.3, linecolor='#333',
        ax=ax, square=True
    )
    ax.set_title(f'{name}', fontsize=13)
    ax.tick_params(labelsize=6)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Semantic Attack Confusion Matrices', fontsize=15, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'semantic_confusion_matrices.png', dpi=200,
            bbox_inches='tight')
plt.show()
print("\u2705 Confusion matrix figures saved.")

## 19. Attack Comparison Bar Chart

In [ ]:
# ════════════════════════════════════════════════════════════════
# ATTACK COMPARISON — GROUPED BAR CHART (with CIs + baseline)
# ════════════════════════════════════════════════════════════════

attack_names = ['Latent Interp.', 'Feat. Distrib.', 'Proto Shift', 'Noise-Denoise']
attack_keys = ['latent_interpolation', 'feature_distribution', 'prototype_shift', 'noise_denoise']
x = np.arange(len(attack_names))
width = 0.22
colors = {'VGG19': '#e74c3c', 'ResNet50': '#3498db', 'DenseNet121': '#2ecc71'}

fig, ax = plt.subplots(figsize=(10, 6))

for i, name in enumerate(MODEL_NAMES):
    rates = [aggregate_metrics[name]['per_attack_fooling_rates'][k] for k in attack_keys]
    # Error bars from bootstrap CIs
    cis = [bootstrap_cis[name][k] for k in attack_keys]
    yerr_lo = [r - ci['ci_lo'] for r, ci in zip(rates, cis)]
    yerr_hi = [ci['ci_hi'] - r for r, ci in zip(rates, cis)]

    bars = ax.bar(x + i * width, [r * 100 for r in rates], width,
                  yerr=[[lo * 100 for lo in yerr_lo], [hi * 100 for hi in yerr_hi]],
                  capsize=3, label=name, color=colors[name], edgecolor='white',
                  linewidth=0.5, error_kw={'linewidth': 1})
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2.5,
                f'{rate*100:.1f}', ha='center', va='bottom', fontsize=8)

# Inversion-only baseline line (mean across models)
mean_inv_baseline = np.mean([inversion_quality[n]['inversion_baseline_fooling'] for n in MODEL_NAMES])
ax.axhline(y=mean_inv_baseline * 100, color='gray', linestyle='--', alpha=0.7,
           label=f'Inversion baseline ({mean_inv_baseline*100:.1f}%)')

ax.set_xlabel('Attack Type', fontsize=12)
ax.set_ylabel('Fooling Rate (%)', fontsize=12)
ax.set_title('Generative Attack Fooling Rates by Model (95% CI)', fontsize=14)
ax.set_xticks(x + width)
ax.set_xticklabels(attack_names)
ax.legend(fontsize=9)
ax.set_ylim(0, 100)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'attack_comparison_bar.png', dpi=200)
plt.show()
print("\u2705 Attack comparison chart saved (with CIs and baseline).")


## 20. Latent Interpolation — Fooling Rate vs Alpha

In [ ]:
# ════════════════════════════════════════════════════════════════
# FOOLING RATE vs ALPHA (LATENT INTERPOLATION)
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))
markers = {'VGG19': 'o', 'ResNet50': 's', 'DenseNet121': 'D'}

for name in MODEL_NAMES:
    rates = [latent_interp_results[name][str(a)]['fooling_rate'] * 100
             for a in INTERP_ALPHAS]
    ax.plot(INTERP_ALPHAS, rates, marker=markers[name], label=name,
            color=colors[name], linewidth=2, markersize=8)

ax.set_xlabel(r'Interpolation $\alpha$', fontsize=12)
ax.set_ylabel('Fooling Rate (%)', fontsize=12)
ax.set_title('Latent Interpolation: Fooling Rate vs Alpha', fontsize=14)
ax.legend()
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'latent_interp_alpha_curve.png', dpi=200)
plt.show()
print("\u2705 Latent interpolation alpha curve saved.")

## 21. Noise-Denoise Perturbation — Fooling Rate vs Step

In [ ]:
# ════════════════════════════════════════════════════════════════
# FOOLING RATE vs DIFFUSION STEP
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))
steps_range = list(range(DIFFUSION_STEPS + 1))

for name in MODEL_NAMES:
    rates = [diffusion_results[name]['per_step'][s]['fooling_rate'] * 100
             for s in steps_range]
    ax.plot(steps_range, rates, marker=markers[name], label=name,
            color=colors[name], linewidth=2, markersize=8)

ax.set_xlabel('Diffusion Step', fontsize=12)
ax.set_ylabel('Fooling Rate (%)', fontsize=12)
ax.set_title('Diffusion-like Noise Attack: Fooling Rate vs Step', fontsize=14)
ax.legend()
ax.set_ylim(0, 100)
ax.set_xticks(steps_range)
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'diffusion_step_curve.png', dpi=200)
plt.show()
print("\u2705 Diffusion step curve saved.")

## 22. Prototype Shift — Fooling Rate vs Lambda

In [ ]:
# ════════════════════════════════════════════════════════════════
# FOOLING RATE vs LAMBDA (PROTOTYPE SHIFT)
# ════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5))

for name in MODEL_NAMES:
    rates = [proto_shift_results[name][str(l)]['fooling_rate'] * 100
             for l in PROTO_LAMBDAS]
    ax.plot(PROTO_LAMBDAS, rates, marker=markers[name], label=name,
            color=colors[name], linewidth=2, markersize=8)

ax.set_xlabel(r'Shift strength $\lambda$', fontsize=12)
ax.set_ylabel('Fooling Rate (%)', fontsize=12)
ax.set_title('Prototype Shift: Fooling Rate vs Lambda', fontsize=14)
ax.legend()
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'proto_shift_lambda_curve.png', dpi=200)
plt.show()
print("\u2705 Prototype shift lambda curve saved.")

---

# PART 4 — SAVE RESULTS

---

## 23. Save All Results

In [ ]:
# ════════════════════════════════════════════════════════════════
# SAVE  genai_results.json
# ════════════════════════════════════════════════════════════════

def make_serializable(obj):
    """Recursively convert numpy types for JSON serialisation."""
    if isinstance(obj, dict):
        return {str(k): make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_serializable(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.bool_):
        return bool(obj)
    return obj


full_results = {
    'metadata': {
        'seed': SEED,
        'test_samples': len(test_paths),
        'num_classes': NUM_CLASSES,
        'models': MODEL_NAMES,
        'max_samples_per_attack': MAX_SAMPLES_PER_ATTACK,
        'interp_alphas': INTERP_ALPHAS,
        'proto_lambdas': PROTO_LAMBDAS,
        'diffusion_steps': DIFFUSION_STEPS,
        'diffusion_sigma': DIFFUSION_SIGMA,
        'inversion_steps': INVERSION_STEPS,
        'inversion_lr': INVERSION_LR
    },
    'inversion_quality': inversion_quality,
    'aggregate_metrics': aggregate_metrics,
    'semantic_structure_scores': sss_genai,
    'bootstrap_confidence_intervals': bootstrap_cis,
    'proto_shift_control': proto_shift_control,
    'latent_interpolation': latent_interp_results,
    'feature_distribution_matching': style_transfer_results,
    'prototype_shift': proto_shift_results,
    'noise_denoise': diffusion_results
}

json_path = RESULTS_DIR / 'genai_results.json'
with open(json_path, 'w') as f:
    json.dump(make_serializable(full_results), f, indent=2)

print(f"\u2705 Full results saved to {json_path}")
print(f"   File size: {json_path.stat().st_size / 1024:.1f} KB")


## 24. Save Combined Semantic Confusion Matrix

In [ ]:
# ════════════════════════════════════════════════════════════════
# COMBINED CONFUSION MATRIX  (sum over all models)
# ════════════════════════════════════════════════════════════════

combined_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int32)
for name in MODEL_NAMES:
    cm = np.load(RESULTS_DIR / f'{name}_semantic_confusion_matrix.npy')
    combined_cm += cm

np.save(RESULTS_DIR / 'semantic_confusion_matrix.npy', combined_cm)
print(f"\u2705 Combined semantic confusion matrix saved.")
print(f"   Shape: {combined_cm.shape}")
print(f"   Total misclassifications: {combined_cm.sum()}")
print(f"   Non-zero entries: {(combined_cm > 0).sum()}")

## 25. Summary Table

In [ ]:
# ════════════════════════════════════════════════════════════════
# FINAL SUMMARY TABLE (with CIs, SSS, no meaningless aggregate)
# ════════════════════════════════════════════════════════════════

def fmt_ci(model, attack_key):
    """Format rate with 95% CI."""
    ci = bootstrap_cis[model][attack_key]
    return f"{ci['rate']*100:.1f} [{ci['ci_lo']*100:.0f}, {ci['ci_hi']*100:.0f}]"

summary_rows = []
for name in MODEL_NAMES:
    m = aggregate_metrics[name]
    summary_rows.append({
        'Model': name,
        'Clean Acc (%)': f"{baselines['models'][name]['accuracy']*100:.1f}",
        'Inv. Baseline (%)': f"{m['inversion_baseline_fooling']*100:.1f}",
        'Latent Interp (%)': fmt_ci(name, 'latent_interpolation'),
        'Feat Distrib (%)': fmt_ci(name, 'feature_distribution'),
        'Proto Shift (%)': fmt_ci(name, 'prototype_shift'),
        'Noise-Denoise (%)': fmt_ci(name, 'noise_denoise'),
        'SSS': f"{sss_genai[name]['sss']:.3f}",
        'Min Lambda': f"{m['min_semantic_distance']['mean_min_lambda_for_flip']:.3f}"
            if m['min_semantic_distance']['mean_min_lambda_for_flip'] is not None else 'N/A',
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

df_summary.to_csv(RESULTS_DIR / 'genai_summary.csv', index=False)
print(f"\n\u2705 Summary table saved to {RESULTS_DIR / 'genai_summary.csv'}")


## 26. Verification Checks

In [ ]:
# ════════════════════════════════════════════════════════════════
# VERIFICATION
# ════════════════════════════════════════════════════════════════

print("Verification Checks")
print("=" * 60)

required_files = [
    RESULTS_DIR / 'genai_results.json',
    RESULTS_DIR / 'genai_per_class_heatmap.csv',
    RESULTS_DIR / 'semantic_confusion_matrix.npy',
    RESULTS_DIR / 'inversion_quality.json',
    RESULTS_DIR / 'genai_semantic_structure_scores.csv',
    RESULTS_DIR / 'genai_top_confused_pairs.json',
    RESULTS_DIR / 'noise_denoise_sensitivity.json',
    RESULTS_DIR / 'genai_summary.csv',
]
for fpath in required_files:
    exists = fpath.exists()
    status = '\u2705' if exists else '\u274c'
    print(f"  {status} {fpath.name}")

# JSON validates
with open(RESULTS_DIR / 'genai_results.json', 'r') as f:
    loaded = json.load(f)
assert loaded['metadata']['seed'] == SEED
assert 'inversion_quality' in loaded
assert 'semantic_structure_scores' in loaded
assert 'bootstrap_confidence_intervals' in loaded
assert 'proto_shift_control' in loaded
print("  \u2705 genai_results.json loads and contains all new fields")

# Confusion matrix
cm_loaded = np.load(RESULTS_DIR / 'semantic_confusion_matrix.npy')
assert cm_loaded.shape == (NUM_CLASSES, NUM_CLASSES)
print(f"  \u2705 Confusion matrix shape: {cm_loaded.shape}")

# SSS values
sss_df = pd.read_csv(RESULTS_DIR / 'genai_semantic_structure_scores.csv')
assert len(sss_df) == len(MODEL_NAMES)
print(f"  \u2705 SSS scores computed for {len(sss_df)} models")

# Inversion quality
inv_q = json.load(open(RESULTS_DIR / 'inversion_quality.json'))
for name in MODEL_NAMES:
    assert name in inv_q
    pres = inv_q[name]['classification_preservation_rate']
    print(f"  \u2705 {name} inversion preservation: {pres*100:.1f}%")

# Heatmap
df_check = pd.read_csv(RESULTS_DIR / 'genai_per_class_heatmap.csv')
for name in MODEL_NAMES:
    assert name in df_check['model'].values
print(f"  \u2705 Heatmap CSV has all models ({len(df_check)} rows)")

print("  \u2705 Inference only — no model weights were modified")

print("\n" + "=" * 60)
print("ALL CHECKS PASSED")
print("=" * 60)

print(f"\nOutput directory: {RESULTS_DIR.resolve()}")
for f in sorted(RESULTS_DIR.rglob('*')):
    if f.is_file():
        print(f"  {f.name:50s} {f.stat().st_size/1024:8.1f} KB")
